In [1]:
# pip install "transformers>=4.43" datasets tokenizers accelerate evaluate sentencepiece
from pathlib import Path
import json

import pandas as pd
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, processors, decoders
from transformers import PreTrainedTokenizerFast

In [2]:
#### 1.
data_dir = Path("data") / "ch13"
data_dir.mkdir(parents=True, exist_ok=True)

In [3]:
#### 2. dataset
train_df = pd.read_csv(Path("data") / "tokenizer" / "train.dataset.tsv", sep="\t")
val_df = pd.read_csv(Path("data") / "tokenizer" / "validation.dataset.tsv", sep="\t")

train_df = train_df.loc[train_df['lang'] == 'en', :]
val_df = val_df.loc[val_df['lang'] == 'en', :]

with open(data_dir / "all.txt", 'w') as f:
    f.write('\n'.join(train_df['text'].apply(lambda x: x.strip()).to_list()))
    f.write('\n')
    f.write('\n'.join(val_df['text'].apply(lambda x: x.strip()).to_list()))
    f.write('\n')

train_df[['text']].to_json(
    data_dir / "train.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)
# jsonl: {"text": "Hello, world and 2025!"}\n{"text": "42"}....

#with open(data_dir / "train.jsonl", 'w') as f:
#    for d in train_df[['text']].to_dict("records"):
#        f.write(json.dumps(d) + "\n")

val_df[['text']].to_json(
    data_dir / "val.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

In [4]:
#### 3. tokenizer
tok = Tokenizer(models.BPE(unk_token="[UNK]"))
tok.pre_tokenizers = pre_tokenizers.ByteLevel(add_prefix_space=True)

trainer = trainers.BpeTrainer(
    vocab_size=32000,
    special_tokens=["[BOS]", "[EOS]", "[PAD]", "[UNK]"],
)

In [5]:
#### 4. train
tok.train([str(data_dir / "all.txt")], trainer)
tok.decoder = decoders.ByteLevel()

bos_id = tok.token_to_id("[BOS]")
eos_id = tok.token_to_id("[EOS]")

tok.post_processor = processors.TemplateProcessing(
    single="[BOS] $A [EOS]",
    pair="[BOS] $A [EOS] [BOS] $B [EOS]",
    special_tokens=[("[BOS]", bos_id), ("[EOS]", eos_id)],
)

# tok.save("tok/tokenizer.json")

In [6]:
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tok,
    #tokenizer_file="tok/tokenizer.json",
    bos_token="[BOS]",
    eos_token="[EOS]",
    pad_token="[PAD]",
    unk_token="[UNK]",

    padding_side="right",
)

tokenizer_dir = data_dir / "tokenizer"
tokenizer_dir.mkdir(parents=True, exist_ok=True)
tokenizer.save_pretrained(tokenizer_dir)

('data/ch13/tokenizer/tokenizer_config.json',
 'data/ch13/tokenizer/special_tokens_map.json',
 'data/ch13/tokenizer/tokenizer.json')

In [7]:
text = "Hello, world!"
tokens = tokenizer.encode(text, add_special_tokens=False)
decoded_text = tokenizer.decode(tokens)

print(f"text={repr(text)}")
print(f"tokens={tokens}")
print(f"decoded_text={repr(decoded_text)}")

text='Hello, world!'
tokens=[18823, 9]
decoded_text='Hello, world!'


In [8]:
#### 5. use tokenizer from huggingface
#from transformers import AutoTokenizer

#tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")

#tokenizer.pad_token = tokenizer.eos_token
#tokenizer.padding_side = "right"